# Basin lab — rollout corpus → clusters → transitions → basins

Empirical basin discovery for the **base / dark / clinical-depression** organisms:

1. **Corpus** — 48 prompts × 8 samples × 3 organisms, 60-token rollouts at temp 0.8
2. **Cluster** — embed endpoints (bge-base), HDBSCAN (KMeans fallback)
3. **Transitions** — at depths t ∈ {12,24,36,48}, resample 8 continuations per probe point and classify where they land
4. **Metastability** — merge clusters that trade trajectories, commitment curves, final basin map

Needs a GPU runtime (A100 recommended). One organism on GPU at a time; total run ≈ 45–90 min.

In [ ]:
import os
if not os.path.exists("/content/dt_rl"):
    !git clone https://github.com/ChuloIva/dt_rl.git /content/dt_rl
%cd /content/dt_rl
!git pull
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    print("no HF_TOKEN from userdata:", e)
%pip install -q -U transformers accelerate sentencepiece sentence-transformers scikit-learn

## Step 1 — rollout corpus (~15–30 min: 3 model loads + 1152 rollouts)

In [ ]:
!python scripts/basin_corpus.py --out data/basin_corpus
!wc -l data/basin_corpus/rollouts_*.jsonl

## Step 2 — cluster the endpoints

In [ ]:
!python scripts/basin_cluster.py --corpus data/basin_corpus
from IPython.display import Markdown, Image, display
display(Image("data/basin_corpus/clusters/scatter.png"))
display(Markdown(open("data/basin_corpus/clusters/report.md").read()))

## Step 3 — perturb & resample (~20–40 min)

For each of the first 4 rollouts per prompt, at depths 12/24/36/48, resample 8 continuations
from the frozen prefix and classify their endpoints with the step-2 cluster model.

In [ ]:
!python scripts/basin_transitions.py --corpus data/basin_corpus
!wc -l data/basin_corpus/transitions/transitions.jsonl

## Step 4 — metastability: which clusters are real basins?

In [ ]:
!python scripts/basin_metastability.py --corpus data/basin_corpus
from IPython.display import Markdown, Image, display
display(Image("data/basin_corpus/basins/commitment.png"))
display(Markdown(open("data/basin_corpus/basins/report.md").read()))

## Save results

Zips everything (rollouts, clusters, transitions, basins) for download or Drive.
The corpus token ids are what Build 4's value head will teacher-force later — keep them.

In [ ]:
!zip -qr basin_results.zip data/basin_corpus
!ls -lh basin_results.zip
# from google.colab import files; files.download("basin_results.zip")
# or: from notebooks.colab_setup import mount_drive; d = mount_drive(); !cp basin_results.zip {d}/results/